In [ ]:
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
MODEL_ID = "answerdotai/ModernBERT-base"
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 3e-5

# Define your label mapping (adjust if your dataset uses different strings)
LABEL_MAP = {
    "not_entailment": 0,
    "entailment": 1
}

In [ ]:
# ==========================================
# 2. LOAD AND CLEAN THE DATASET
# ==========================================
print("Loading dataset...")
# Assuming you have CSV files. If TSV, add delimiter='\t'
dataset = load_dataset(
    'csv',
    data_files={
        'train': 'train.csv',
        'validation': 'dev.csv'
    }
)

# Rename columns to match what the tokenizer expects (if necessary)
if "sentence1" in dataset["train"].column_names:
    dataset = dataset.rename_column("sentence1", "premise")
    dataset = dataset.rename_column("sentence2", "hypothesis")

# # Convert string labels to integers using our LABEL_MAP
# def map_labels(example):
#     example['label'] = LABEL_MAP.get(example['label'], -1) # -1 catches weird data
#     return example

# dataset = dataset.map(map_labels)
# # Filter out any rows that had invalid labels (where label became -1)
# dataset = dataset.filter(lambda x: x['label'] in [0, 1])

Loading dataset...


In [ ]:
# ==========================================
# 3. TOKENIZATION
# ==========================================
print("Tokenizing dataset...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def preprocess_function(examples):
    # Pass premise and hypothesis together; the tokenizer handles the [SEP] tokens
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

tokenized_datasets = dataset.map(preprocess_function, batched=True)

Tokenizing dataset...


Map:   0%|          | 0/24432 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

In [ ]:
# ==========================================
# 4. DEFINE METRICS (ACCURACY & F1)
# ==========================================
print("Loading evaluation metrics...")
acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Calculate Accuracy
    accuracy = acc_metric.compute(predictions=predictions, references=labels)["accuracy"]

    # Calculate F1 Score (average="binary" is standard for 2-class tasks)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="binary")["f1"]

    return {
        "accuracy": accuracy,
        "f1": f1
    }

Loading evaluation metrics...


In [ ]:
# ==========================================
# 5. INITIALIZE MODEL & TRAINER
# ==========================================
print("Initializing ModernBERT...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label={0: "NOT_ENTAILMENT", 1: "ENTAILMENT"},
    label2id={"NOT_ENTAILMENT": 0, "ENTAILMENT": 1}
)

training_args = TrainingArguments(
    output_dir="./modernbert-rte-model",
    eval_strategy="epoch",          # Evaluate at the end of each epoch
    save_strategy="epoch",            # Save a model checkpoint each epoch
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2, # Eval can handle larger batches
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,      # Automatically keep the best performing model
    metric_for_best_model="f1",       # Use F1 to determine the "best" model, not loss
    bf16=True,                        # Set to False if you are on an older GPU without mixed precision
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,    # <--- CHANGE THIS LINE (used to be tokenizer=tokenizer)
    compute_metrics=compute_metrics
)

Initializing ModernBERT...


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# ==========================================
# 6. TRAIN AND EVALUATE
# ==========================================
print("Starting training loop...")
trainer.train()

print("Evaluating final model on validation set...")
final_results = trainer.evaluate()
print(f"Final Validation Accuracy: {final_results['eval_accuracy']:.4f}")
print(f"Final Validation F1 Score: {final_results['eval_f1']:.4f}")

# Save the final fine-tuned model and tokenizer so you can use it later!
trainer.save_model("./modernbert-rte-final")
tokenizer.save_pretrained("./modernbert-rte-final")
print("Training complete and model saved!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Starting training loop...


W0321 09:51:56.089000 4636 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.273312,0.288424,0.883314,0.885590
2,0.162048,0.442138,0.891479,0.897576
3,0.037020,0.582067,0.893112,0.897582


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating final model on validation set...


Final Validation Accuracy: 0.8931
Final Validation F1 Score: 0.8976


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete and model saved!
